# ingest large CSV

## 4.1. read paths

In [0]:
ROOT_PATH = "/Volumes/workspace/bronze/logistic_landing"

display(dbutils.fs.ls(ROOT_PATH))

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/bronze/logistic_landing/batch_large"
    )
)

In [0]:
BASE_PATH = (
    "/Volumes/workspace/bronze/logistic_landing/batch_large"
)

CSV_PATH = f"{BASE_PATH}/shipments_100k.csv"

print("CSV files:")
display(dbutils.fs.ls(f"{CSV_PATH}"))


## 4.2. Check number of rows

In [0]:
csv_count = (
    spark.read
    .option("header", "true")
    .csv(CSV_PATH)
    .count()
)

print(f"CSV rows: {csv_count:,}")

assert csv_count == 100_000

## 4.3 Ingest large csv file into a table

In [0]:

import time
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, LongType
)

TARGET_TABLE = "workspace.bronze.shipments_large_csv"

schema = StructType([
    StructField("shipment_id", StringType()),
    StructField("distribution_center", StringType()),
    StructField("store_id", StringType()),
    StructField("planned_delivery", DateType()),
    StructField("actual_delivery", DateType()),
    StructField("ordered_units", LongType()),
    StructField("delivered_units", LongType()),
])

start = time.perf_counter()

csv_df = (
    spark.read
    .option("header", "true")
    .option("mode", "FAILFAST")
    .option("dateFormat", "yyyy-MM-dd")
    .schema(schema)
    .csv(CSV_PATH)
)

(
    csv_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

elapsed = time.perf_counter() - start

print(f"CSV ingestion time: {elapsed:.2f} seconds")
print(f"Target: {TARGET_TABLE}")
